# 07 - Wrapper Feature Selection

This notebook performs the fourth classification experiment using a
wrapper-based feature-selection strategy.

Unlike filter methods, wrapper methods evaluate feature subsets according to
the predictive performance of a classification algorithm.

In this experiment, the scikit-learn classifiers use Sequential Feature
Selection with a Decision Tree to choose the subset. RIPPER uses a bounded
forward search with the same outer evaluation protocol.

Each classifier is evaluated using its selected subset, the same outer
cross-validation folds and evaluation metrics, and its established
preprocessing representation.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.feature_selection import SequentialFeatureSelector

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

pd.set_option("display.max_columns", None)

from pa1_utils import evaluate_ripper, summarize_ripper, export_ripper_folds

## 1. Dataset Preparation

The same dataset preparation used in the previous experiments is maintained.

Identifier and constant attributes are removed, while `Attrition` is encoded
as a binary target:

- `No` -> 0
- `Yes` -> 1

Keeping the dataset preparation unchanged allows the effect of the wrapper
strategy to be evaluated under the same experimental conditions.

In [ ]:
DATA_PATH = "../../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_PATH)

columns_to_remove = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df_model = df.drop(columns=columns_to_remove)

X = df_model.drop(columns=["Attrition"])

y = df_model["Attrition"].map({
    "No": 0,
    "Yes": 1
})

print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)

## 2. Preprocessing Definition

The model-specific preprocessing used in the previous experiments is
preserved. Numerical predictors are standardized and categorical predictors
are one-hot encoded for the scikit-learn classifiers within each outer
training fold. RIPPER uses original predictors and training-fold numerical
discretization.

In [ ]:
numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical predictors:", len(numerical_features))
print("Categorical predictors:", len(categorical_features))

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

## 3. Wrapper Feature Selection

Wrapper methods select features according to the predictive performance
obtained by a classifier.

Sequential Forward Selection is used in this experiment.

The procedure starts with an empty feature set and iteratively adds the feature
that provides the largest improvement in classification performance.

A Decision Tree is used as the estimator responsible for evaluating candidate
feature subsets.

This wrapper retains 15 transformed features for the scikit-learn classifiers,
matching the dimensionality used in the filter experiments. RIPPER selects
original predictors with its own search budget.

In [ ]:
N_SELECTED_FEATURES = 15

print(
    f"Number of transformed features to retain: "
    f"{N_SELECTED_FEATURES}"
)

## 4. Wrapper Estimator

A Decision Tree is used as the estimator inside the Sequential Feature
Selector.

The estimator is kept simple and deterministic by fixing its random state.

This selector is used for K-NN, Decision Tree, SVM and Neural Network.
RIPPER uses the classifier-specific forward search described below.

In [ ]:
wrapper_estimator = DecisionTreeClassifier(
    random_state=42
)

## 5. Evaluation Strategy

The same stratified 10-fold cross-validation procedure used in the previous
experiments is maintained.

For each scikit-learn classifier and outer cross-validation fold:

1. Preprocessing is fitted using only the training data.
2. Sequential Feature Selection is applied using only the training data.
3. Fifteen transformed features are selected.
4. The classifier is trained using the selected features.
5. The learned preprocessing and selected subset are applied to the
   validation fold.
6. Performance is measured exclusively on the validation fold.

RIPPER also selects features using only the outer training fold, with the
search configuration described below. Held-out data do not influence either
selection procedure.

In [ ]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

## 6. Evaluation Metrics

The same evaluation metrics used in the previous experiments are retained:

- Accuracy
- Precision
- Recall
- F1-score
- Balanced Accuracy

Particular attention is given to recall, F1-score, and balanced accuracy due
to the imbalance of the target variable.

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy"
}

## 7. Classification Algorithms

The same classifier configurations used in the previous experiments are
maintained.

This ensures that the feature-selection strategy remains the main
experimental factor that changes.

In [ ]:
models = {
    "K-NN": KNeighborsClassifier(),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "SVM": SVC(),

    "Neural Network": MLPClassifier(
        random_state=42,
        max_iter=2000
    )
}

## 8. Evaluation Function

For K-NN, Decision Tree, SVM and Neural Network, preprocessing, wrapper
feature selection, and classification are combined into a scikit-learn
`Pipeline`.

Sequential Feature Selection is performed independently within each training
fold, preventing information leakage.

In [ ]:
def evaluate_model(
    name,
    classifier,
    X,
    y,
    cv,
    scoring
):
    """
    Evaluate a classifier using preprocessing,
    wrapper feature selection, and stratified
    cross-validation.
    """

    feature_selector = SequentialFeatureSelector(
        estimator=wrapper_estimator,
        n_features_to_select=N_SELECTED_FEATURES,
        direction="forward",
        scoring="f1",
        cv=5,
        n_jobs=1
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessing", preprocessor),
            (
                "feature_selection",
                feature_selector
            ),
            ("classifier", classifier)
        ]
    )

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=1
    )

    result = {
        "Model": name,
        "Selected Features": N_SELECTED_FEATURES
    }

    for metric in scoring.keys():

        metric_values = scores[f"test_{metric}"]

        result[f"{metric}_mean"] = metric_values.mean()
        result[f"{metric}_std"] = metric_values.std()

    return result

## 9. Model Evaluation

Each classifier is evaluated after wrapper-based feature selection.

Because Sequential Feature Selection evaluates many candidate subsets, this
experiment is computationally more expensive than the filter-based
experiments.

### Model-specific wrapper: RIPPER

RIPPER uses a classifier-specific forward search over original
predictors, with stratified two-fold inner CV, F1 and at most three additions.
The best prefix is retained; ties favour fewer predictors. Every candidate
refits RIPPER and its numerical bins on the inner training split. Outer
validation data never enter selection. This budget differs from the
tree-based 15-component, five-fold wrapper. For the tree-based wrapper, scaling
and encoding are fitted on the outer training data before inner CV; its inner
scores have this representation limitation, while outer evaluation stays held out.

RIPPER uses seed 42 and the same outer folds and metrics as the other
classifiers.

In [ ]:
results = []

for name, model in models.items():

    print(f"Evaluating {name}...")

    result = evaluate_model(
        name=name,
        classifier=model,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring
    )

    results.append(result)

print("\nEvaluation completed.")

In [ ]:
ripper_folds = pd.DataFrame(evaluate_ripper(
    experiment='Wrapper', X=X, y=y, cv=cv, preprocessor=preprocessor
))
results.append(summarize_ripper(ripper_folds))


## 10. Complete Cross-Validation Results

The following table reports the mean and standard deviation of each evaluation
metric across the ten outer cross-validation folds.

In [ ]:
wrapper_results = pd.DataFrame(results)

display(
    wrapper_results.round(4)
)

## 11. Wrapper Selection Performance Summary

The following table summarizes the mean predictive performance obtained after
wrapper-based feature selection.

In [ ]:
summary_results = wrapper_results[
    [
        "Model",
        "Selected Features",
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
        "f1_mean",
        "balanced_accuracy_mean"
    ]
].copy()

summary_results.columns = [
    "Model",
    "Selected Features",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Balanced Accuracy"
]

summary_results = summary_results.sort_values(
    "F1",
    ascending=False
)

display(
    summary_results.round(4)
)

## 12. Inspection of the Selected Features

After completing the honest nested cross-validation experiment, preprocessing
and Sequential Feature Selection are fitted to the complete dataset solely for
interpretation.

This additional fit is not used to estimate predictive performance.

Its purpose is to identify the transformed features selected by the wrapper
when all available data are considered.

In [ ]:
inspection_selector = SequentialFeatureSelector(
    estimator=wrapper_estimator,
    n_features_to_select=N_SELECTED_FEATURES,
    direction="forward",
    scoring="f1",
    cv=5,
    n_jobs=1
)

inspection_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "feature_selection",
            inspection_selector
        )
    ]
)

inspection_pipeline.fit(X, y)

In [ ]:
fitted_preprocessor = (
    inspection_pipeline
    .named_steps["preprocessing"]
)

fitted_selector = (
    inspection_pipeline
    .named_steps["feature_selection"]
)

all_feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

selected_mask = (
    fitted_selector
    .get_support()
)

selected_feature_names = (
    all_feature_names[selected_mask]
)

print(
    f"Selected features: "
    f"{len(selected_feature_names)}"
)

display(
    pd.Series(
        selected_feature_names,
        name="Selected transformed feature"
    )
)

In [ ]:
wrapper_feature_selection = pd.DataFrame({
    "Feature": all_feature_names,
    "Selected": selected_mask
})

wrapper_feature_selection = (
    wrapper_feature_selection
    .sort_values(
        "Selected",
        ascending=False
    )
)

display(
    wrapper_feature_selection.head(20)
)

## 13. Export Results

The wrapper cross-validation results, summary table, and selected feature
information are stored for the final comparison between feature-selection
strategies.

In [ ]:
wrapper_results.to_csv(
    "../../results/pa1/wrapper_results.csv",
    index=False
)

summary_results.to_csv(
    "../../results/pa1/wrapper_summary.csv",
    index=False
)

wrapper_feature_selection.to_csv(
    "../../results/pa1/wrapper_selected_features.csv",
    index=False
)

print(
    "Wrapper feature-selection results saved successfully."
)

# Preserve other strategies in the dedicated rules/bins/fold artifact.
export_ripper_folds(ripper_folds, "../../results/pa1/ripper_fold_results.csv")

## 14. Comparison with Previous Experiments

The wrapper results are compared with the baseline, univariate filter, and
multivariate filter experiments.

All experiments use the same dataset, classifiers, outer cross-validation
folds, and evaluation metrics. Each classifier retains its stated
representation and wrapper configuration.

In [ ]:
baseline_summary = pd.read_csv(
    "../../results/pa1/baseline_summary.csv"
)

univariate_summary = pd.read_csv(
    "../../results/pa1/univariate_summary.csv"
)

multivariate_summary = pd.read_csv(
    "../../results/pa1/multivariate_summary.csv"
)

baseline_comparison = baseline_summary.copy()
baseline_comparison["Experiment"] = "All Features"

univariate_comparison = univariate_summary.copy()
univariate_comparison["Experiment"] = "Univariate"

multivariate_comparison = multivariate_summary.copy()
multivariate_comparison["Experiment"] = "Multivariate"

wrapper_comparison = summary_results.copy()
wrapper_comparison["Experiment"] = "Wrapper"

comparison = pd.concat(
    [
        baseline_comparison,
        univariate_comparison,
        multivariate_comparison,
        wrapper_comparison
    ],
    ignore_index=True
)

comparison = comparison[
    [
        "Experiment",
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Balanced Accuracy"
    ]
]

display(
    comparison.round(4)
)

## 15. Main Observations

The tree-based wrapper reduces the scikit-learn representation from 51 to
15 transformed features. RIPPER selects a mean of 2.6 original predictors
across outer folds.

The results show that the effect of wrapper selection depends considerably on
the classifier:

- Among the classifiers using the tree-based wrapper, the **Neural Network** achieves the strongest performance after wrapper
  selection, with an F1-score of 0.3919 and a balanced accuracy of 0.6286.
  However, this remains clearly below its baseline performance using all
  features (F1 = 0.4738; balanced accuracy = 0.6765).

- The **Decision Tree** obtains an F1-score of 0.3594 and a balanced accuracy
  of 0.6205. Its performance is slightly lower than its baseline result and
  also lower than the results obtained with the univariate and multivariate
  filters.

- **K-NN benefits from wrapper selection compared with the original feature
  representation.** Its F1-score increases from 0.2115 in the baseline to
  0.3146, while recall increases from 0.1350 to 0.2364. Among the three
  feature-selection strategies, the wrapper provides the best K-NN result.

- The **SVM** retains relatively high accuracy (0.8456) and precision
  (0.6638), but its recall falls to 0.0971. Consequently, its F1-score is only
  0.1672. This again illustrates why accuracy alone is insufficient for this
  imbalanced classification problem.

- **RIPPER** obtains F1-score 0.3996 and balanced accuracy 0.6557, the
  highest scores among reduced-feature configurations. Its wrapper selects
  two or three original predictors per fold.

- Compared with the two filter approaches, wrapper selection produces the
  highest F1-score for K-NN and a similar Neural Network performance to the
  multivariate filter. However, it does not improve the best overall results
  obtained using the complete feature representation.

- The feature subset selected for interpretation includes variables related
  to age, income, job involvement, professional experience, business travel,
  job role, marital status, and overtime. Several of these characteristics
  were also identified by the previous feature-selection strategies and
  exploratory analysis.

Overall, wrapper-based feature selection substantially reduces the number of
features and improves K-NN performance, but it does not outperform the best
baseline model. The Neural Network using all transformed features remains the
strongest configuration according to F1-score and balanced accuracy.

### RIPPER Rule Interpretation

Each row below belongs to an outer training-fold model, rather than a single
full-data model. A matching conjunction predicts attrition; unmatched cases
predict no attrition. Inspect conditions in their recorded units and use the
exported bins when interpreting intervals. These associations are not causal.
The artifact also retains selected inputs and the wrapper trace for each fold.

In [ ]:
display(ripper_folds[["Fold", "Number of Features", "Selected Features", "Rules"]])
print("Example: rules from outer training fold 1")
print(ripper_folds.iloc[0]["Rules"])
display(ripper_folds[["Accuracy", "Precision", "Recall", "F1", "Balanced Accuracy"]].mean().to_frame("RIPPER mean"))